In [13]:
import pandas as pd
import seaborn as sns
taxis = sns.load_dataset("taxis")
taxis["pickup"] = pd.to_datetime(taxis["pickup"])
taxis["dropoff"] = pd.to_datetime(taxis["dropoff"])
print(taxis.shape)  
print(taxis["pickup"].min())
print(taxis["pickup"].max())

(6433, 14)
2019-02-28 23:29:03
2019-03-31 23:43:45


In [14]:
par_jour = (
    taxis.set_index("pickup")
    .resample("D")
    .agg(nb_courses=("total", "size"), chiffre_affaires=("total", "sum"))
)
par_jour.head(10)

,nb_courses,chiffre_affaires
pickup,,
2019-02-28,1,6.30
2019-03-01,241,4213.83
2019-03-02,198,3319.02
2019-03-03,169,3027.32
2019-03-04,171,3269.08
2019-03-05,228,4377.38
2019-03-06,257,4673.61
2019-03-07,218,4115.70
2019-03-08,235,4524.21


In [15]:
taxis["heure"] = taxis["pickup"].dt.hour
courses_par_heure = taxis["heure"].value_counts().sort_index()
heure_la_plus_chargee = courses_par_heure.idxmax()
print(f"Heure la plus chargée : {heure_la_plus_chargee}h ({courses_par_heure.max()} courses)")
courses_par_heure

Heure la plus chargée : 18h (417 courses)


heure
0     205
1     110
2     101
3      67
4      57
5      51
6     142
7     221
8     313
9     319
10    325
11    295
12    334
13    316
14    360
15    330
16    336
17    388
18    417
19    406
20    368
21    355
22    321
23    296
Name: count, dtype: int64

In [16]:
taxis["jour_semaine"] = taxis["pickup"].dt.day_name()
pourboire_moyen_jour = taxis.groupby("jour_semaine")["tip"].mean().round(2).sort_values(ascending=False)
print(f"Meilleur jour : {pourboire_moyen_jour.index[0]}")
pourboire_moyen_jour

Meilleur jour : Wednesday


jour_semaine
Wednesday    2.09
Thursday     2.06
Monday       2.03
Tuesday      2.00
Friday       1.98
Sunday       1.91
Saturday     1.82
Name: tip, dtype: float64

In [17]:
taxis["duree_min"] = (taxis["dropoff"] - taxis["pickup"]).dt.total_seconds() / 60
print(taxis["duree_min"].describe())
negatives = taxis[taxis["duree_min"] < 0]
trop_longues = taxis[taxis["duree_min"] > 180]
print("négatives :", len(negatives))
print("> 3 h :", len(trop_longues))
trop_longues[["pickup", "dropoff", "distance", "duree_min"]].head()

count    6433.000000
mean       14.349617
std        11.643892
min         0.000000
25%         6.500000
50%        10.900000
75%        18.516667
max       107.666667
Name: duree_min, dtype: float64
négatives : 0
> 3 h : 0


,pickup,dropoff,distance,duree_min


In [18]:
nb_avant = len(taxis)
taxis = taxis[taxis["duree_min"] > 0]
print(f"{nb_avant - len(taxis)} courses retirées, {len(taxis)} restantes")

6 courses retirées, 6427 restantes


aucune durée négative aucune course > 3h et les courses de 0min avaient aussi une distance de 0 donc cest des courses annulées ou erreur donc retirées